# Импорт зависимостей

In [5]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.downloader import download_warc_files
from src.data.converter import convert_all_warc
from src.data.cleaner import clean_all_jsonl
from src.data.entropy import compute_dataset_entropy_dir, filter_dataset_dir
import src.tokenization.tokenizers as tk


# 3.1 Скачать датасет

## 1. Скачать часть Comon Crawl - WARC файлы

In [2]:
download_warc_files(config_path="../configs/config.yaml", raw_data_dir="../data/raw")

17:17:18 | INFO     | Директория для сохранения: D:\Folders\Master Degree\labs\MNNA-2026-labs-DmitrievDM\MNNA-2026-labs-DmitrievDM\data\raw


Общий прогресс:   0%|          | 0/1 [00:00<?, ?it/s]

17:17:18 | INFO     | Начинаем скачивание: CC-MAIN-20260605214811-20260606004811-00000.warc.gz


CC-MAIN-20260605214811-20260606004811-00000.warc.gz:   0%|          | 0.00/940M [00:00<?, ?B/s]

17:17:48 | ERROR    | Ошибка сети при скачивании https://data.commoncrawl.org/crawl-data/CC-MAIN-2026-25/segments/1780687572080.85/warc/CC-MAIN-20260605214811-20260606004811-00000.warc.gz: HTTPSConnectionPool(host='data.commoncrawl.org', port=443): Read timed out.


## 2. Конверитровать WARC файлы в текстовый формат.

In [3]:
convert_all_warc(input_dir="../data/raw", output_dir="../data/converted")

17:28:53 | INFO     | Найдено WARC-файлов: 2


Конвертация WARC:   0%|          | 0/2 [00:00<?, ?file/s]

17:28:53 | INFO     | Пропуск ..\data\raw\CC-MAIN-20260605214811-20260606004811-00000.warc.gz: результат уже существует
17:28:53 | INFO     | Пропуск ..\data\raw\CC-MAIN-20260605214811-20260606004811-00001.warc.gz: результат уже существует
17:28:53 | INFO     | Обработка завершена
17:28:53 | INFO     | Файлов найдено: 2
17:28:53 | INFO     | Сконвертировано: 0
17:28:53 | INFO     | Пропущено: 2
17:28:53 | INFO     | Ошибок: 0


# 3.2 Очистка данных

In [4]:
clean_all_jsonl(input_dir="../data/converted", output_dir="../data/cleaned")

10:45:56 | INFO     | Обработка: ..\data\converted\CC-MAIN-20260605214811-20260606004811-00000.jsonl
11:53:06 | INFO     | Файл ..\data\converted\CC-MAIN-20260605214811-20260606004811-00000.jsonl обработан: прочитано=20799, оставлено объектов=9947, отброшено=10852, итоговых чанков записано=22295
11:53:06 | INFO     | Обработка: ..\data\converted\CC-MAIN-20260605214811-20260606004811-00001.jsonl
12:38:12 | INFO     | Файл ..\data\converted\CC-MAIN-20260605214811-20260606004811-00001.jsonl обработан: прочитано=20768, оставлено объектов=9815, отброшено=10953, итоговых чанков записано=21700


***Примечания:*** *стоит отладить код, добавить прогресс-бар, посмотреть как работает фильтрация и нужна ли она, также выяснить причины очень долгого выполнения очистки данных*

# 3.3 Улучшить качество данных

## 1. GPT2 и энтропия

In [ ]:
stats = compute_dataset_entropy_dir(
    input_dir="../data/cleaned",
    output_dir="../data/entropy",
    stats_path="../data/stats/entropy_stats.json",
    pattern="*.jsonl",
    model_name="gpt2",
    text_field="text",
    batch_size=8,
    max_length=1024,
)

print(stats["info_density_nats_per_token"])

19:09:30 | INFO     | Using device: cpu


19:09:31 | INFO     | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
19:09:31 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
19:09:31 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:09:31 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
19:09:32 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
19:09:32 | INFO     | HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
19:09:32 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

19:09:34 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
19:09:34 | INFO     | Processing file: ..\data\cleaned\CC-MAIN-20260605214811-20260606004811-00000.jsonl


CC-MAIN-20260605214811-20260606004811-00000.jsonl: 0batch [00:00, ?batch/s]

## 2. Удаление дубликатов и объектов с высокой или низкой энтропией

In [ ]:
filter_dataset_dir(
    original_dir="../data/cleaned",               # Папка с ИСХОДНЫМ датасетом
    metrics_dir="../data/entropy",    # Папка, куда предыдущий код сохранил энтропию
    output_dir="../data/cleaned_final",       # Папка, куда сохранится ИТОГОВЫЙ чистый датасет
    lower_percentile=1.0,                  # Удалить 1% текстов с самой НИЗКОЙ энтропией
    upper_percentile=99.0                  # Удалить 1% текстов с самой ВЫСОКОЙ энтропией
)

# 3.4 Токенизация

In [ ]:
from pathlib import Path
import random
import json

# =========================
# 1. Настройки
# =========================
# Папка, куда filter_dataset_dir сохранил очищенные .jsonl файлы
FILTERED_DATA_DIR = Path("../data/cleaned_final") 

TEXT_FIELD = "text"
SEED = 42
OUTPUT_DIR = Path("../artifacts/tokenizers")

# Если данных очень много и не хватает оперативной памяти (RAM), 
# ограничьте количество текстов. Например, MAX_TEXTS = 200_000.
# Если хотите читать всё до конца, поставьте None.
MAX_TEXTS = 200_000 

# =========================
# 2. Простой цикл для чтения всех .jsonl файлов
# =========================
print(f"Поиск .jsonl файлов в {FILTERED_DATA_DIR}...")
jsonl_files = sorted(FILTERED_DATA_DIR.glob("*.jsonl"))

if not jsonl_files:
    raise FileNotFoundError(f"Не найдено .jsonl файлов в {FILTERED_DATA_DIR}")

print(f"Найдено файлов: {len(jsonl_files)}")
print("Чтение текстов...")

texts = []
for file_path in jsonl_files:
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            try:
                obj = json.loads(line)
                text = obj.get(TEXT_FIELD, "")
                if text:
                    texts.append(str(text))
            except json.JSONDecodeError:
                continue
                
            # Прерываем чтение, если достигли лимита
            if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
                break
                
    if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
        print(f"Достигнут лимит в {MAX_TEXTS} текстов. Останавливаем чтение.")
        break

print(f"Итого загружено текстов: {len(texts)}")
assert len(texts) > 0, "Список текстов пуст!"

# Выбираем случайный объект для демонстрации
rng = random.Random(SEED)
sample_idx = rng.randrange(len(texts))
sample_text = texts[sample_idx]

print("-" * 60)
print(f"Индекс случайного объекта: {sample_idx}")
print(f"Пример текста (первые 150 символов): {sample_text[:150]}")

Загрузка данных из data\processed\cleaned_common_crawl.jsonl...


FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\cleaned_common_crawl.jsonl'

## 3.4.1 Символьная токенизация

In [ ]:
print("-" * 60)
print("Обучение символьного токенизатора...")
char_token2id = tk.fit_char_tokenizer(texts, max_samples=None, seed=SEED)
char_ids = tk.encode_char(sample_text, char_token2id, add_special=True)

print("1. Токенизация по символам")
print(f"Размер словаря: {len(char_token2id)}")
print(f"Размер последовательности случайного объекта: {len(char_ids)}")
# Сохранение
char_path = tk.save_vocab(char_token2id, OUTPUT_DIR / "char_tokenizer.json")

## 3.4.2 Токенизация по словам

In [ ]:
print("-" * 60)
print("Обучение словесного токенизатора...")
word_token2id = tk.fit_word_tokenizer(
    texts,
    max_samples=100_000, 
    max_vocab_size=30_000,
    min_freq=2,
    seed=SEED
)
word_ids = tk.encode_word(sample_text, word_token2id, add_special=True)

print("2. Токенизация по словам")
print(f"Размер словаря: {len(word_token2id)}")
print(f"Размер последовательности случайного объекта: {len(word_ids)}")
# Сохранение
word_path = tk.save_vocab(word_token2id, OUTPUT_DIR / "word_tokenizer.json")

## 3.4.3 BPE

In [ ]:
print("-" * 60)
print("Обучение BPE токенизатора (может занять пару минут)...")
bpe_tokenizer = tk.train_bpe_tokenizer(
    texts,
    max_samples=100_000,
    vocab_size=10_000,
    min_freq=2,
    seed=SEED
)
bpe_ids = tk.encode_bpe(sample_text, bpe_tokenizer, add_special=True)

print("3. BPE-токенизация")
print(f"Размер словаря: {bpe_tokenizer.get_vocab_size()}")
print(f"Размер последовательности случайного объекта: {len(bpe_ids)}")
# Сохранение
bpe_path = tk.save_bpe_tokenizer(bpe_tokenizer, OUTPUT_DIR / "bpe_tokenizer.json")